# DS4 Agent-Based Modeling (ABM) -- Stage 4 Step 1

**Notebook:** `57_ds4_agent_based.ipynb`  
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_abm_blueprint.md`

---

## Paper grounding (OneSec paper SS2.7)

> "The value of ABM is that it can test whether simple local rules are sufficient to reproduce broader observed patterns... Empirical features from Information Theory, Stochastic Processes, and Network Theory can be used to inform the agent types and rules."

## Validation target -- DS4's own open question, not ported from Structured

Structured's ABM tested reproduction of a Fano-factor burstiness target -- a count-process-specific statistic that does not apply to DS4's mean-reverting OU process. DS4's own StoP notebook found a real, unresolved divergence instead: median daily-fitted relaxation time tau_hat~=0.63 days vs median weekly-fitted tau_hat~=0.91 weeks, only r=0.316 correlation between daily and weekly theta_hat per participant. This notebook tests whether a pure daily-OU agent -- the simplest possible agent, built only from each participant's own already-fitted (mu,theta,sigma) -- reproduces this divergence, or predicts tighter daily/weekly consistency than what real participants show.

## Output

```
outputs/model_outputs/ds4_agent_based/
  participant_abm_simulation_results.csv
  abm_model_comparison_summary.csv
  model_run_summary.md
  figures/
```


## FLAG decisions

| Flag | Decision |
|---|---|
| FLAG 1 | Agent parameters = each real participant's own fitted (mu,theta,sigma) from StoP -- not re-estimated |
| FLAG 2 | Model 1 (baseline): pure fixed-parameter discrete OU agent |
| FLAG 3 | Model 2: Model 1 + a weekly-periodic drift modulation (illustrative amplitude=0.1, NOT fitted to real day-of-week data -- a simplifying assumption, same spirit as Structured's own fixed 3-day session-length assumption) |
| FLAG 4 | Validation = compare simulated daily-vs-weekly theta divergence to StoP's own real numbers |
| FLAG 5 | Simulated series length = each participant's real n_days_obs, gap-free (simplification vs real data's actual gaps) |
| FLAG 6 | Simulated X_t clipped to [0,1] after each step -- the OU model does not otherwise respect the real quantity's physical bounds (same caveat as StoP/StM's "OU model-implied, not literal" language) |


## DS4 structural cautions

- Only `fit_valid=True` StoP participants (641) can be simulated -- inherits StoP's own eligibility population.
- This is a recoverability/sufficiency test, not a claim participants literally follow an OU process.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = Path("outputs/model_outputs/ds4_agent_based")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
MIN_PAIRS = 10
MIN_WEEKS = 5


In [ ]:
stop = pd.read_csv("outputs/model_outputs/ds4_stochastic_processes/participant_parameters.csv")
stop_weekly = pd.read_csv(
    "outputs/model_outputs/ds4_stochastic_processes/participant_parameters_weekly_sensitivity.csv")

valid = stop[stop["fit_valid"]].merge(
    stop_weekly[["participant_id_canonical", "theta_hat_OU_weekly", "tau_hat_weekly", "fit_valid_weekly"]],
    on="participant_id_canonical", how="left",
)
print(f"Valid StoP fits (agent population): {len(valid)}")
print(f"...of which also weekly-valid (real target available): {int(valid['fit_valid_weekly'].sum())}")


In [ ]:
def fit_ar1(x_t, x_t1):
    if len(x_t) < MIN_PAIRS:
        return np.nan, np.nan
    X = np.column_stack([np.ones_like(x_t), x_t])
    coef, *_ = np.linalg.lstsq(X, x_t1, rcond=None)
    c, phi = coef
    if not (0 < phi < 1):
        return np.nan, np.nan
    theta = -np.log(phi)
    return theta, phi


def simulate_ou_path(mu, theta, sigma, n_days, rng, weekly_amplitude=0.0):
    """Model 1 if weekly_amplitude=0; Model 2 otherwise (adds a day-of-week
    drift modulation with fixed, non-fitted amplitude -- FLAG 3)."""
    phi = np.exp(-theta)
    cond_var = sigma ** 2 * (1 - phi ** 2) / (2 * theta)
    cond_std = np.sqrt(max(cond_var, 0))
    x = np.empty(n_days)
    x[0] = np.clip(mu, 0, 1)
    for t in range(1, n_days):
        mu_t = mu
        if weekly_amplitude > 0:
            mu_t = mu * (1 + weekly_amplitude * np.cos(2 * np.pi * t / 7))
            mu_t = np.clip(mu_t, 0.01, 0.99)
        x[t] = mu_t + (x[t - 1] - mu_t) * phi + rng.normal(0, cond_std)
        x[t] = np.clip(x[t], 0.0, 1.0)
    return x


def refit_daily_and_weekly(x, min_weeks=MIN_WEEKS):
    n = len(x)
    theta_daily, _ = fit_ar1(x[:-1], x[1:])
    n_weeks = n // 7
    theta_weekly = np.nan
    if n_weeks >= min_weeks:
        weekly = np.array([x[i * 7:(i + 1) * 7].mean() for i in range(n_weeks)])
        theta_weekly, _ = fit_ar1(weekly[:-1], weekly[1:])
    return theta_daily, theta_weekly


In [ ]:
rng = np.random.RandomState(RANDOM_STATE)

results = []
for _, row in valid.iterrows():
    n_days = int(row["n_days_obs"])
    if n_days < 14:
        continue
    mu, theta, sigma = row["mu_hat_OU"], row["theta_hat_OU"], row["sigma_hat_OU"]

    x_m1 = simulate_ou_path(mu, theta, sigma, n_days, rng, weekly_amplitude=0.0)
    theta_d_m1, theta_w_m1 = refit_daily_and_weekly(x_m1)

    x_m2 = simulate_ou_path(mu, theta, sigma, n_days, rng, weekly_amplitude=0.1)
    theta_d_m2, theta_w_m2 = refit_daily_and_weekly(x_m2)

    results.append(dict(
        participant_id_canonical=row["participant_id_canonical"], n_days_obs=n_days,
        real_theta_daily=theta, real_tau_daily=row["tau_hat"],
        real_theta_weekly=row["theta_hat_OU_weekly"], real_tau_weekly=row["tau_hat_weekly"],
        model1_theta_daily=theta_d_m1, model1_theta_weekly=theta_w_m1,
        model2_theta_daily=theta_d_m2, model2_theta_weekly=theta_w_m2,
    ))

sim = pd.DataFrame(results)
print(f"Simulated {len(sim)} agents (Model 1 and Model 2)")


In [ ]:
def divergence_stats(theta_daily, theta_weekly, tau_daily=None, tau_weekly=None, label=""):
    valid_pair = theta_daily.notna() & theta_weekly.notna()
    n = int(valid_pair.sum())
    corr = theta_daily[valid_pair].corr(theta_weekly[valid_pair])
    if tau_daily is not None:
        med_daily = tau_daily[valid_pair].median()
        med_weekly = tau_weekly[valid_pair].median()
    else:
        med_daily = (1 / theta_daily[valid_pair]).median()
        med_weekly = (1 / theta_weekly[valid_pair]).median()
    print(f"{label}: n={n}, corr(theta_daily,theta_weekly)={corr:.4f}, "
          f"median tau_daily={med_daily:.3f}, median tau_weekly(in weeks)={med_weekly:.3f}")
    return dict(n=n, corr=corr, median_tau_daily=med_daily, median_tau_weekly=med_weekly)


print("=== REAL (StoP's own numbers) ===")
real_stats = divergence_stats(valid["theta_hat_OU"], valid["theta_hat_OU_weekly"],
    valid["tau_hat"], valid["tau_hat_weekly"], "Real")

print("\n=== MODEL 1 (pure daily OU, no added mechanism) ===")
m1_stats = divergence_stats(sim["model1_theta_daily"], sim["model1_theta_weekly"], label="Model 1")

print("\n=== MODEL 2 (+ weekly-periodic drift, amplitude=0.1) ===")
m2_stats = divergence_stats(sim["model2_theta_daily"], sim["model2_theta_weekly"], label="Model 2")


In [ ]:
comparison = pd.DataFrame([
    {"model": "real", **real_stats},
    {"model": "model1_pure_OU", **m1_stats},
    {"model": "model2_weekly_periodic", **m2_stats},
])
comparison["daily_weekly_ratio"] = comparison["median_tau_weekly"] / comparison["median_tau_daily"]
print(comparison)


In [ ]:
sim.to_csv(OUT_DIR / "participant_abm_simulation_results.csv", index=False)
comparison.to_csv(OUT_DIR / "abm_model_comparison_summary.csv", index=False)
print("Wrote participant_abm_simulation_results.csv, abm_model_comparison_summary.csv")


In [ ]:
summary_lines = [
    "# DS4 ABM -- Model Run Summary", "",
    "**Notebook:** `57_ds4_agent_based.ipynb`  ",
    "**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_abm_blueprint.md`  ",
    "**Run date:** " + pd.Timestamp.now().strftime("%Y-%m-%d"), "",
    f"Agent population: {len(sim)}", "",
    "## Real vs Model 1 vs Model 2", "", "```",
    comparison.to_string(), "```",
]
with open(OUT_DIR / "model_run_summary.md", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print("Wrote model_run_summary.md")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
real_tau_w = valid.loc[valid["fit_valid_weekly"] == True, "tau_hat_weekly"].dropna()
m1_tau_w = (1 / sim["model1_theta_weekly"]).dropna()
m2_tau_w = (1 / sim["model2_theta_weekly"]).dropna()
ax.hist(real_tau_w, bins=30, alpha=0.5, label=f"real (n={len(real_tau_w)})", color="#4C72B0", density=True)
ax.hist(m1_tau_w, bins=30, alpha=0.5, label=f"Model 1 (n={len(m1_tau_w)})", color="#C44E52", density=True)
ax.hist(m2_tau_w, bins=30, alpha=0.5, label=f"Model 2 (n={len(m2_tau_w)})", color="#55A868", density=True)
ax.set_xlabel("weekly-fitted tau_hat (weeks)")
ax.set_ylabel("density")
ax.set_title("DS4 ABM: real vs simulated weekly relaxation time")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_ds4_abm_weekly_tau_comparison.png", dpi=150)
plt.close(fig)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (theta_d_col, theta_w_col, label) in zip(axes, [
    ("model1_theta_daily", "model1_theta_weekly", "Model 1"),
    ("model2_theta_daily", "model2_theta_weekly", "Model 2"),
]):
    d = sim[[theta_d_col, theta_w_col]].dropna()
    ax.scatter(d[theta_d_col], d[theta_w_col], s=10, alpha=0.5, color="#8172B2")
    r = d[theta_d_col].corr(d[theta_w_col])
    ax.set_xlabel("simulated daily theta_hat")
    ax.set_ylabel("simulated weekly theta_hat")
    ax.set_title(f"{label} (r={r:.3f}; real r=0.316)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_ds4_abm_daily_vs_weekly_scatter.png", dpi=150)
plt.close(fig)


## Interpretation and limits

**Read the cells above for live numbers.**

- Compare the `daily_weekly_ratio` and `corr` columns across real/Model 1/Model 2 in the comparison table. If Model 1 (pure OU, no added mechanism) already shows a ratio and correlation close to the real numbers, the observed daily/weekly divergence is a natural statistical consequence of aggregating a mean-reverting process, not evidence of deeper behavioral complexity. If Model 1 shows much tighter daily-weekly consistency (ratio close to 1, correlation close to 1) than the real data, that is evidence real behavior has additional structure a pure daily-OU agent does not capture -- and Model 2's result then indicates whether a simple non-fitted weekly-periodic mechanism can close some of that gap.
- Model 2's amplitude (0.1) is illustrative, not fitted to real day-of-week patterns -- a simplifying assumption, same spirit as Structured's own fixed 3-day session-length assumption. A negative result for Model 2 does not rule out other mechanisms.
- This is a recoverability/sufficiency test of the OU model assumption, not a claim participants literally follow an OU process.


## Forward connections

This closes the DS4 model-family arc: all 8 CAS model families now have some form of DS4 implementation (5 originally active + GaT + ABM revisited after Step 2; SyD remains structurally excluded -- no repeated survey waves in DS4, reserved for DS6).
